<a href="https://colab.research.google.com/github/Man980/py_project/blob/main/fds_capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FDS - CAPSTONE PROJECT

In [ ]:
# Load libraries
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# INFOS PERSONNELLES
#
# Le nom du groupe
# Nom des etudiants
# Prof
# Date
# Nom du programme
# ...

# Structure du script

Le script est organisé en plusieurs phases successives, chacune répondant à un objectif précis dans le processus de détection de fraude.

* PHASE 0 — Chargement et exploration : importation et analyse préliminaire des données.
* PHASE 1 — Contrôle de cohérence : vérification de la qualité et bold textde la cohérence des données.
* PHASE 2 — Isolation de `isFraud` : mise à l'écart de la variable cible pour une approche non supervisée.
* PHASE 3 — Feature engineering : création de nouvelles variables pertinentes.
* PHASE 4 — Encodage et mise à l'échelle : préparation des données pour les algorithmes.
* PHASE 5 — Isolation Forest : détection des anomalies globales.
* PHASE 6 — Local Outlier Factor (LOF) : détection des anomalies locales.
* PHASE 7 — DBSCAN : identification des clusters et des points isolés.
* PHASE 8 — Analyse de graphe : détection de schémas frauduleux complexes.
* PHASE 9 — Score de suspicion : agrégation des résultats en un score unique.
* PHASE 10 — Validation : comparaison avec `isFraud` pour évaluer les performances *(optionnelle)*.


In [ ]:
# ==============================================================================
# PHASE 0 : CHARGEMENT ET EXPLORATION
# ==============================================================================

path = "/content/drive/MyDrive/mobile_money_synthetic_dataset.csv"
df = pd.read_csv(path)

print("="* 70)
print("PHASE 0: EXPLORATION")
print("="* 70)
print("Dimentions : ", df.shape)
print("\nTypes de colonnes :\n", df.dtypes)
print("Valeurs manquantes :\n", df.isnull().sum())
print("Doublons sur ID : ", df["ID"].duplicated().sum())
print("Doublons sur toutes les colonnes :", df.duplicated().sum())

for col in ["account_type", "region", "transact_type", "day_of_week"]:
  print(f"\n--- Modalites de {col} ---")
  print(df[col].value_counts())

print("\n Statistiques Descriptives \n", df.describe())

Dimentions :  (1000000, 17)

Types de colonnes :
 ID                      object
Transact_amount        float64
cash_in                  int64
cash_out                 int64
hour                     int64
month                    int64
day_of_week             object
account_type            object
name_orig               object
name_dest               object
sender_bal_before      float64
sender_bal_after       float64
receiver_bal_before    float64
receiver_bal_after     float64
region                  object
transact_type           object
isFraud                  int64
dtype: object
Valeurs manquantes :
 ID                     0
Transact_amount        0
cash_in                0
cash_out               0
hour                   0
month                  0
day_of_week            0
account_type           0
name_orig              0
name_dest              0
sender_bal_before      0
sender_bal_after       0
receiver_bal_before    0
receiver_bal_after     0
region                 0
transact_typ

In [ ]:
# ==============================================================================
# PHASE 1 : CONTROLE DE COHERENCE DES DONNEES
# ==============================================================================

print("="* 70)
print("PHASE 1: CONTROLE DE COHERENCE")
print("="* 70)

# Equation comptable emetteur : solde_avant - montant = solde apres
df["sender_check"] = df["sender_bal_before"] - df["Transact_amount"] - df["sender_bal_after"]
# Equation comptable destinataire : solde_avant + montant = solde apres
df["receiver_check"] = df["receiver_bal_before"] + df["Transact_amount"] - df["receiver_bal_after"]

TOLERANCE = O.01
####### WE HAVE TO GET BACK IN THIS BLOCK FOR REVIEW ########
df["sender_incoherent"] = df["sender_check"].abs() > TOLERANCE
df["receiver_incoherent"] = df["receiver_check"].abs() > TOLERANCE

print("Incoherence emetteur : ", df["sender_incoherent"].sum()) # Difference entre ce block df sans parenthese
print("Incoherence destinataire : ", df["receiver_incoherent"].sum())

print("Soldes negatifs (sender_bal_before) : ", (df["sender_bal_before"] < 0).sum()) # et ce block et df avec parenthese
print("Soldes negatifs (sender_bal_after) :", (df["sender_bal_after"] < 0).sum())
print("Soldes negatifs (receiver_bal_before) :", (df["receiver_bal_before"] < 0).sum())
print("Soldes negatifs (receiver_bal_after) :", (df["receiver_bal_after"] < 0).sum())

PLAFOND LEGAL = 100_000

print("Transactions au-dessus du plafond legal (100 000 HTG) :", (df["transact_amount"] > PLAFOND_LEGAL).sum())

print("Heures hors plage (0-23) :", ((df["hour"] < 0) | (df["hour"] > 23)).sum())
print("Mois hors plage (1-12) :", ((df["month"] < 1) | (df["month"] > 12)).sum())



In [ ]:
# ==============================================================================
# PHASE 2 : ISOLATION DE isFraud (mise a l'ecart, non utilisee pour la suite)
# ==============================================================================

# On sauvegarde isFraud a part, puis on la retire completement du dataframe
# de travail pour etre certain qu'elle ne "fuite" jamais dans les features
# labels_is_Fraud = df["isFraud"].copy()
df_work = df.drop(columns = ["isFraud"]).copy()

print("\n isFraud isolee dans labels_isFraud. Colonnes restantes : \n", df_work.shape[1])


In [ ]:
# ==============================================================================
# PHASE 3 : FEATURE ENGINEERING
# ==============================================================================

print("\n" _ "=" * 70)
print("PHASE 3 : FEATURE ENGINEERING")
print("=" * 70)

# --- 3.1 Proximite du plafond legal : plus Transact_amount se rapproche
#    de  100 000 HTG, plus ce ratio se rapproche de 1 (indice de smurfing)
df_work["ratio_plafond"] = df_work["Transact_amount"] / PLAFOND_LEGAL

#--- 3.2 Frequence de Transactions par compte emetteur et destinataire
#  (proxy de volume d'activite ; sans horodatage precis avec annee,
#   on utilise le nombre total d'occurrences dans le dataset)

freq_orig = df_work["name_orig"].value_counts()
freq_dest = df_work["name_dest"].value_counts()

df_work["freq_orig"] = df_work["name_orig"].map(freq_orig)
df_work["freq_dest"] = df_work["name_dest"].map(freq_dest)

#--- 3.3 Montant total envoye / recu par compte (volume cumule)
total_sent = df_work.groupby("name_orig")[""]



# --------- THIS PART CONCERNS THE FIRST MANIPULATIONS -----------

In [ ]:
df["name_orig"].value_counts()

In [ ]:
df[["transact_type", "cash_in", "cash_out"]].value_counts()

In [ ]:
df.head(50)

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 17 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   ID                   1000000 non-null  object 
 1   Transact_amount      1000000 non-null  float64
 2   cash_in              1000000 non-null  int64  
 3   cash_out             1000000 non-null  int64  
 4   hour                 1000000 non-null  int64  
 5   month                1000000 non-null  int64  
 6   day_of_week          1000000 non-null  object 
 7   account_type         1000000 non-null  object 
 8   name_orig            1000000 non-null  object 
 9   name_dest            1000000 non-null  object 
 10  sender_bal_before    1000000 non-null  float64
 11  sender_bal_after     1000000 non-null  float64
 12  receiver_bal_before  1000000 non-null  float64
 13  receiver_bal_after   1000000 non-null  float64
 14  region               1000000 non-null  object 
 15 

In [ ]:
# ==================================================
# Générateur Dataset Mobile Money Synthétique
# ==================================================

import numpy as np
import pandas as pd


# ==================================================
# Paramètres
# ==================================================

np.random.seed(42)

N_TRANSACTIONS = 1_000_000
N_ACCOUNTS = 100_000



# ==================================================
# 1. Génération des comptes
# ==================================================

accounts = pd.DataFrame({

    "account_id": [
        "ACC_" + str(i).zfill(7)
        for i in range(N_ACCOUNTS)
    ]

})


# -------------------------------
# Type de compte
# -------------------------------

account_types = [
    "Personnel",
    "Marchand",
    "Agent",
    "Entreprise"
]


accounts["account_type"] = np.random.choice(
    account_types,
    size=N_ACCOUNTS,
    p=[
        0.75,
        0.15,
        0.08,
        0.02
    ]
)



# -------------------------------
# Région
# -------------------------------

regions = [
    "Port-au-Prince",
    "Delmas",
    "Petion-Ville",
    "Kenscoff",
    "Cite Soleil",
    "Tabarre",
    "Carrefour",
    "Gressier"
]


prob_region = [
    0.45,
    0.15,
    0.10,
    0.08,
    0.07,
    0.05,
    0.06,
    0.04
]


accounts["region"] = np.random.choice(
    regions,
    size=N_ACCOUNTS,
    p=prob_region
)



# ==================================================
# 2. Génération des transactions
# ==================================================

df = pd.DataFrame()


# Identifiant transaction

df["ID"] = [
    "TX_" + str(i).zfill(8)
    for i in range(N_TRANSACTIONS)
]



# Type de transaction

transaction_types = [
    "CASH_IN",
    "CASH_OUT",
    "TRANSFER",
    "PAYMENT"
]


df["transact_type"] = np.random.choice(
    transaction_types,
    size=N_TRANSACTIONS,
    p=[
        0.25,
        0.25,
        0.35,
        0.15
    ]
)



# ==================================================
# 3. Comptes origine et destination
# ==================================================

df["name_orig"] = np.random.choice(
    accounts["account_id"],
    size=N_TRANSACTIONS
)


df["name_dest"] = np.random.choice(
    accounts["account_id"],
    size=N_TRANSACTIONS
)


# Eviter les transactions vers soi-même

mask = df["name_orig"] == df["name_dest"]

df.loc[mask, "name_dest"] = np.random.choice(
    accounts["account_id"],
    size=mask.sum()
)



# ==================================================
# 4. Variables temporelles
# ==================================================

dates = (
    pd.Timestamp("2025-01-01")
    +
    pd.to_timedelta(
        np.random.randint(
            0,
            365*24,
            N_TRANSACTIONS
        ),
        unit="h"
    )
)


df["hour"] = dates.hour

df["month"] = dates.month

df["day_of_week"] = dates.day_name()



# ==================================================
# 5. Montants des transactions
# ==================================================

df["Transact_amount"] = np.random.lognormal(
    mean=8,
    sigma=1.2,
    size=N_TRANSACTIONS
)


df["Transact_amount"] = (
    df["Transact_amount"]
    .round(2)
)



# ==================================================
# 6. Soldes avant transaction
# ==================================================

df["sender_bal_before"] = (
    np.random.lognormal(
        mean=10,
        sigma=1,
        size=N_TRANSACTIONS
    )
    .round(2)
)



df["receiver_bal_before"] = (
    np.random.lognormal(
        mean=10,
        sigma=1,
        size=N_TRANSACTIONS
    )
    .round(2)
)



# Empêcher un retrait supérieur au solde

df["Transact_amount"] = np.minimum(
    df["Transact_amount"],
    df["sender_bal_before"]
)



# Solde après transaction

df["sender_bal_after"] = (
    df["sender_bal_before"]
    -
    df["Transact_amount"]
)



df["receiver_bal_after"] = (
    df["receiver_bal_before"]
    +
    df["Transact_amount"]
)



# ==================================================
# 7. Variables CASH
# ==================================================

df["cash_in"] = (
    df["transact_type"]
    ==
    "CASH_IN"
).astype(int)



df["cash_out"] = (
    df["transact_type"]
    ==
    "CASH_OUT"
).astype(int)



# ==================================================
# 8. Informations comptes
# ==================================================

account_info = accounts.set_index(
    "account_id"
)


df["account_type"] = (
    df["name_orig"]
    .map(account_info["account_type"])
)


df["region"] = (
    df["name_orig"]
    .map(account_info["region"])
)



# ==================================================
# 9. Création du label fraude
# ==================================================

df["isFraud"] = 0



# Cas 1 : transactions exceptionnellement élevées

threshold = df["Transact_amount"].quantile(0.995)


df.loc[
    df["Transact_amount"] > threshold,
    "isFraud"
] = 1



# Cas 2 : grosses transactions la nuit

df.loc[
    (df["hour"] <= 4)
    &
    (df["Transact_amount"] > 50000),
    "isFraud"
] = 1



# Cas 3 : comportement agent suspect

agent_accounts = accounts[
    accounts["account_type"] == "Agent"
]["account_id"]


df.loc[
    (
        df["name_orig"]
        .isin(agent_accounts)
    )
    &
    (
        df["Transact_amount"] > 100000
    ),
    "isFraud"
] = 1



# ==================================================
# 10. Dataset final
# ==================================================

dataset = df[
[
    "ID",
    "Transact_amount",
    "cash_in",
    "cash_out",
    "hour",
    "month",
    "day_of_week",
    "account_type",
    "name_orig",
    "name_dest",
    "sender_bal_before",
    "sender_bal_after",
    "receiver_bal_before",
    "receiver_bal_after",
    "region",
    "transact_type",
    "isFraud"
]
]



# ==================================================
# 11. Export CSV
# ==================================================

path = "/content/drive/MyDrive/mobile_money_synthetic_dataset.csv"


dataset.to_csv(
    path,
    index=False
)



# ==================================================
# Vérifications
# ==================================================

print(dataset.head())

print("\nDimension du dataset :")
print(dataset.shape)


print("\nRépartition fraude :")
print(dataset["isFraud"].value_counts())


print("\nTaux fraude :")
print(round(dataset["isFraud"].mean()*100,2), "%")


print("\nFichier sauvegardé :", path)